# GMA / FDIIR — V2 Reference Implementation

**Version:** 2 (V2)  
**Date:** May 12, 2026  
**Status:** Reference implementation — CCM scale {0, 0.5, 1.0} · Vetor A scale 1-5  

**Estudo Dirigido — Mestrado FDIIR**  

Execução dos quatro métodos de seleção morfológica sobre a CCM:  
Scoring linear ponderado · Ideal point · Pareto-based MA · HMMD  

---

### Overview

This notebook is the reference implementation for Version 2 of the GMA/FDIIR research. It provides a complete walkthrough of:
- Morphological field definition and configuration space generation
- Cross-Consistency Matrix (CCM) with scale {0, 0.5, 1.0}
- Four selection methods applied to 144 admissible configurations
- Comparative analysis and stability metrics

## Setup

In [ ]:
import numpy as np
import pandas as pd
import itertools
from scipy.stats import spearmanr

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

### Data

Project constants.

#### Morphological Parameters and Design Alternatives (DAs)

In [ ]:
# Modules and their values (internal IDs)
MODULES = ['D', 'I', 'ID', 'R']

MODULE_VALUES = {
    'D':  ['D1', 'D2', 'D3', 'D4'],
    'I':  ['I1', 'I2', 'I3', 'I4'],
    'ID': ['ID1', 'ID2', 'ID3'],
    'R':  ['R1', 'R2', 'R3']
}

# Descriptive names of DAs
DA_NAMES = {
    'D1':  'model-based',
    'D2':  'signal-based',
    'D3':  'data-driven',
    'D4':  'active diagnosis',
    'I1':  'output comparison',
    'I2':  'voting',
    'I3':  'dependency modeling',
    'I4':  'adaptive filters',
    'ID1': 'rule-based',
    'ID2': 'statistical',
    'ID3': 'ML identification',
    'R1':  'exclusion',
    'R2':  'weight reconfiguration',
    'R3':  'bypass'
}

# Criteria
CRITERIA = ['C1', 'C2', 'C3', 'C4', 'C5']
CRITERIA_NAMES = {
    'C1': 'Interpretabilidade',
    'C2': 'Custo computacional (inv.)',
    'C3': 'Latência (inv.)',
    'C4': 'Maturidade tecnológica',
    'C5': 'Cobertura de falhas'
}

print("Modules:", MODULES)
print("Total DAs:", sum(len(v) for v in MODULE_VALUES.values()))
print("Raw configuration space:", np.prod([len(v) for v in MODULE_VALUES.values()]), "configs")

#### Vector A (Vetor A)

In [ ]:
# Vector A: scores C1–C5 per DA
# Column order: C1, C2, C3, C4, C5
# Scale: 1-5
VETOR_A_RAW = {
    # Module D
    'D1':  [5, 3, 4, 5, 4],
    'D2':  [4, 4, 5, 5, 3],
    'D3':  [1, 1, 2, 3, 4],
    'D4':  [4, 3, 2, 2, 3],
    # Module I
    'I1':  [5, 3, 4, 5, 4],
    'I2':  [5, 3, 4, 5, 3],
    'I3':  [4, 2, 2, 3, 4],
    'I4':  [3, 3, 4, 4, 4],
    # Module ID
    'ID1': [5, 5, 5, 5, 3],
    'ID2': [3, 3, 4, 5, 4],
    'ID3': [1, 2, 2, 3, 3],
    # Module R
    'R1':  [5, 5, 5, 5, 3],
    'R2':  [4, 3, 3, 4, 4],
    'R3':  [5, 4, 5, 4, 4]
}

# DataFrame of Vector A
vetor_a = pd.DataFrame(VETOR_A_RAW, index=CRITERIA).T
vetor_a.index.name = 'DA'
vetor_a['Sigma'] = vetor_a.sum(axis=1)

print("Vector A — Consolidated Scores (V2, scale 1-5)")
print(vetor_a.to_string())

### CCM — Cross-Consistency Matrix

Scale {0.0, 0.5, 1.0}.  
Result of 3 multi-LLM passes + primary sources.

In [ ]:
# CCM — 6 blocks of pairwise compatibility
# Structure: CCM[(da1, da2)] = score {0.0, 0.5, 1.0}
# Symmetric: CCM[(a,b)] == CCM[(b,a)]

CCM_RAW = {
    # B1: D × R
    ('D1', 'R1'): 1.0, ('D1', 'R2'): 1.0, ('D1', 'R3'): 1.0,
    ('D2', 'R1'): 1.0, ('D2', 'R2'): 1.0, ('D2', 'R3'): 1.0,
    ('D3', 'R1'): 1.0, ('D3', 'R2'): 1.0, ('D3', 'R3'): 1.0,
    ('D4', 'R1'): 0.5, ('D4', 'R2'): 1.0, ('D4', 'R3'): 0.5,

    # B2: D × ID
    ('D1', 'ID1'): 1.0, ('D1', 'ID2'): 1.0, ('D1', 'ID3'): 1.0,
    ('D2', 'ID1'): 1.0, ('D2', 'ID2'): 1.0, ('D2', 'ID3'): 1.0,
    ('D3', 'ID1'): 0.5, ('D3', 'ID2'): 1.0, ('D3', 'ID3'): 1.0,
    ('D4', 'ID1'): 0.5, ('D4', 'ID2'): 0.5, ('D4', 'ID3'): 0.5,

    # B3: D × I
    ('D1', 'I1'): 1.0, ('D1', 'I2'): 1.0, ('D1', 'I3'): 1.0, ('D1', 'I4'): 1.0,
    ('D2', 'I1'): 1.0, ('D2', 'I2'): 1.0, ('D2', 'I3'): 1.0, ('D2', 'I4'): 1.0,
    ('D3', 'I1'): 0.5, ('D3', 'I2'): 1.0, ('D3', 'I3'): 1.0, ('D3', 'I4'): 1.0,
    ('D4', 'I1'): 0.5, ('D4', 'I2'): 0.5, ('D4', 'I3'): 1.0, ('D4', 'I4'): 0.5,

    # B4: I × R
    ('I1', 'R1'): 1.0, ('I1', 'R2'): 1.0, ('I1', 'R3'): 1.0,
    ('I2', 'R1'): 1.0, ('I2', 'R2'): 1.0, ('I2', 'R3'): 1.0,
    ('I3', 'R1'): 1.0, ('I3', 'R2'): 1.0, ('I3', 'R3'): 1.0,
    ('I4', 'R1'): 1.0, ('I4', 'R2'): 1.0, ('I4', 'R3'): 0.5,

    # B5: I × ID
    ('I1', 'ID1'): 1.0, ('I1', 'ID2'): 1.0, ('I1', 'ID3'): 1.0,
    ('I2', 'ID1'): 1.0, ('I2', 'ID2'): 1.0, ('I2', 'ID3'): 1.0,
    ('I3', 'ID1'): 1.0, ('I3', 'ID2'): 1.0, ('I3', 'ID3'): 1.0,
    ('I4', 'ID1'): 1.0, ('I4', 'ID2'): 1.0, ('I4', 'ID3'): 1.0,

    # B6: ID × R
    ('ID1', 'R1'): 1.0, ('ID1', 'R2'): 1.0, ('ID1', 'R3'): 1.0,
    ('ID2', 'R1'): 1.0, ('ID2', 'R2'): 1.0, ('ID2', 'R3'): 0.5,
    ('ID3', 'R1'): 1.0, ('ID3', 'R2'): 1.0, ('ID3', 'R3'): 1.0,
}

# Make symmetric
CCM = {}
for (a, b), v in CCM_RAW.items():
    CCM[(a, b)] = v
    CCM[(b, a)] = v

def get_ccm(da1, da2):
    """Returns CCM score for a DA pair. Returns 1.0 if same module (irrelevant)."""
    if da1 == da2:
        return 1.0
    return CCM.get((da1, da2), CCM.get((da2, da1), None))

# Verification
n_cells = len(CCM_RAW)
n_05 = sum(1 for v in CCM_RAW.values() if v == 0.5)
n_10 = sum(1 for v in CCM_RAW.values() if v == 1.0)
n_00 = sum(1 for v in CCM_RAW.values() if v == 0.0)

print(f"CCM loaded: {n_cells} cells")
print(f"  1.0 (compatible):     {n_10} ({100*n_10/n_cells:.1f}%)")
print(f"  0.5 (conditional):    {n_05} ({100*n_05/n_cells:.1f}%)")
print(f"  0.0 (incompatible):   {n_00} ({100*n_00/n_cells:.1f}%)")

### Scenarios SC1–SC4

Module weights homogeneous across all scenarios (w_D = w_I = w_ID = w_R = 0.25).  
Differentiation exclusively by criteria weights (w_C1–w_C5).

In [ ]:
# Scenarios: module and criteria weights
SCENARIOS = {
    'SC1': {
        'name': 'Uniform',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .20, 'C2': .20, 'C3': .20, 'C4': .20, 'C5': .20}
    },
    'SC2': {
        'name': 'Safety / certification',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .35, 'C2': .10, 'C3': .10, 'C4': .35, 'C5': .10}
    },
    'SC3': {
        'name': 'Coverage / performance',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .20, 'C3': .20, 'C4': .10, 'C5': .40}
    },
    'SC4': {
        'name': 'Resource efficiency',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .35, 'C3': .35, 'C4': .10, 'C5': .10}
    }
}

# Verification: criteria weights sum to 1.0
for sc_id, sc in SCENARIOS.items():
    total = sum(sc['criteria_weights'].values())
    assert abs(total - 1.0) < 1e-9, f"{sc_id}: criteria weights sum to {total}"

# Display
rows = []
for sc_id, sc in SCENARIOS.items():
    row = {'Scenario': sc_id, 'Semantics': sc['name']}
    row.update({f'w_{k}': v for k, v in sc['module_weights'].items()})
    row.update({f'w_{k}': v for k, v in sc['criteria_weights'].items()})
    rows.append(row)

df_scenarios = pd.DataFrame(rows).set_index('Scenario')
print("Scenarios SC1-SC4")
print(df_scenarios.to_string())

## Solution Space

### Configuration Generation

Cartesian product D × I × ID × R

In [ ]:
def generate_configurations():
    """
    Generates all 144 configurations from the Cartesian product D x I x ID x R.
    Returns list of dicts: {'D': da_d, 'I': da_i, 'ID': da_id, 'R': da_r}
    """
    configs = []
    for d, i, id_, r in itertools.product(
        MODULE_VALUES['D'],
        MODULE_VALUES['I'],
        MODULE_VALUES['ID'],
        MODULE_VALUES['R']
    ):
        configs.append({'D': d, 'I': i, 'ID': id_, 'R': r})
    return configs


def compute_w(config):
    """
    Computes w(S) = minimum of pairwise compatibilities between DAs
    from different modules in configuration S.
    Source: Levin (2012), Section 3.7 — N(S) = (w(S); n(S))
    The 6 pairs correspond to blocks B1-B6 of CCM E5.
    """
    das = [config['D'], config['I'], config['ID'], config['R']]
    scores = []
    for da1, da2 in itertools.combinations(das, 2):
        score = get_ccm(da1, da2)
        if score is None:
            raise ValueError(f"Pair not found in CCM: ({da1}, {da2})")
        scores.append(score)
    return min(scores)


# Generate all configurations with w(S)
all_configs = generate_configurations()
for cfg in all_configs:
    cfg['w'] = compute_w(cfg)

print(f"Configurations generated: {len(all_configs)}")
print(f"  w(S) = 1.0: {sum(1 for c in all_configs if c['w'] == 1.0)}")
print(f"  w(S) = 0.5: {sum(1 for c in all_configs if c['w'] == 0.5)}")
print(f"  w(S) = 0.0: {sum(1 for c in all_configs if c['w'] == 0.0)}")

### Admissible Configuration Space Filtering

Admissibility criterion: w(S) >= 0.5.

CCM has zero cells at 0.0 — all configurations have at least conditional compatibility on all pairs.

Threshold w >= 0.5 admits the full space excluding only absolute incompatibilities.

In [ ]:
def filter_admissible(configs, threshold=0.5):
    """
    Filters configurations with w(S) >= threshold.
    Returns list of admissible configurations with sequential ID.
    """
    admissible = [c for c in configs if c['w'] >= threshold]
    for i, cfg in enumerate(admissible, 1):
        cfg['id'] = f"S{i:02d}"
    return admissible


admissible = filter_admissible(all_configs, threshold=0.5)

print(f"Admissible space: {len(admissible)} of {len(all_configs)} configurations")
print(f"Reduction: {100*(1 - len(admissible)/len(all_configs)):.1f}%")
print()

# Table of admissible space
df_admissible = pd.DataFrame([
    {
        'ID': c['id'],
        'D': f"{c['D']} ({DA_NAMES[c['D']]})",
        'I': f"{c['I']} ({DA_NAMES[c['I']]})",
        'ID_': f"{c['ID']} ({DA_NAMES[c['ID']]})",
        'R': f"{c['R']} ({DA_NAMES[c['R']]})",
        'w(S)': c['w']
    }
    for c in admissible
])

print(df_admissible.head(20).to_string(index=False))
print(f"... ({len(admissible) - 20} more rows)")

### Solution Space Summary

In [ ]:
n_total = len(all_configs)
n_admissible = len(admissible)
n_w10 = sum(1 for c in admissible if c['w'] == 1.0)
n_w05 = sum(1 for c in admissible if c['w'] == 0.5)

print("Summary of solution space")
print("-" * 40)
print(f"Raw space:             {n_total} configurations (4 x 4 x 3 x 3)")
print(f"Threshold:             w(S) >= 0.5")
print(f"Admissible space:      {n_admissible} configurations")
print(f"Reduction:             {100*(1 - n_admissible/n_total):.1f}%")
print(f"  w(S) = 1.0:          {n_w10} configurations (no conditional pairs)")
print(f"  w(S) = 0.5:          {n_w05} configurations (at least one conditional pair)")
print()
print("Interpretation: CCM does not eliminate any configuration due to")
print("absolute incompatibility. Differentiation is done by selection methods.")

### Selection Methods

Four methods implemented on the admissible space (144 configurations).

Each method exposes the same interface:
- Input: list of configurations, scenario, Vector A
- Output: ranking or Pareto-efficient set

#### Linear Weighted Scoring

Formulation (Levin 2012, Section 3.5 — Multiple Choice Problem):

$$Q(S) = \sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot \text{score}(da_m, c)$$

Where:
- $w_m$: weight of module $m$ in scenario (uniform = 0.25)
- $w_c$: weight of criterion $c$ in scenario
- $\text{score}(da_m, c)$: score of DA selected for module $m$ on criterion $c$ (Vector A)

Output: scalar $Q(S)$ per configuration — ranking by descending value.

In [ ]:
def scoring_linear(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']
    results = []

    for cfg in admissible:
        q = 0.0
        for mod in MODULES:
            da = cfg[mod]
            da_score = sum(
                w_crit[c] * vetor_a.loc[da, c]
                for c in CRITERIA
            )
            q += w_mod[mod] * da_score
        results.append({'id': cfg['id'], 'Q': round(q, 4)})

    results.sort(key=lambda x: x['Q'], reverse=True)
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Execute for all scenarios
print("Linear Weighted Scoring -- Top 10 per scenario")
print("=" * 60)

scoring_results = {}
for sc_id, sc in SCENARIOS.items():
    res = scoring_linear(admissible, sc, vetor_a)
    scoring_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'Q':>6}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['Q']:>6.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

#### Ideal Point Method

Calculates the distance of each configuration to the ideal point S^o.

$S^o$: for each module, the DA with the highest score weighted by criteria.

$$S^* = \arg\min_{S \in \{S_a\}} \rho(S, S^o)$$

$$\rho(S, S^o) = \sqrt{\sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot (score^o_c - score(da_m, c))^2}$$

Where $S^o$ is the ideal configuration formed by the best DA of each module
by weighted criteria score. Output: distance $\rho$ per configuration —
ranking by ascending value (shorter distance = better).

In [ ]:
def ideal_point(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']

    # Calculate ideal point: best DA per module (highest weighted score)
    ideal = {}
    for mod in MODULES:
        best_da = max(
            MODULE_VALUES[mod],
            key=lambda da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
        )
        ideal[mod] = {c: vetor_a.loc[best_da, c] for c in CRITERIA}

    results = []
    for cfg in admissible:
        rho_sq = 0.0
        for mod in MODULES:
            da = cfg[mod]
            for c in CRITERIA:
                diff = ideal[mod][c] - vetor_a.loc[da, c]
                rho_sq += w_mod[mod] * w_crit[c] * (diff ** 2)
        rho = round(rho_sq ** 0.5, 4)
        results.append({'id': cfg['id'], 'rho': rho})

    results.sort(key=lambda x: x['rho'])
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Execute for all scenarios
print("Ideal Point -- Top 10 per scenario")
print("=" * 60)

ideal_results = {}
for sc_id, sc in SCENARIOS.items():
    res = ideal_point(admissible, sc, vetor_a)
    ideal_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'rho':>7}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['rho']:>7.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

#### Pareto-Based Morphological Analysis

A configuration $S$ is Pareto-efficient if there does not exist $S'$ such that:

$$\forall c \in \text{CRITERIA}: f_c(S') \geq f_c(S) \quad \text{and} \quad \exists c: f_c(S') > f_c(S)$$

Where $f_c(S)$ is the aggregated score of configuration $S$ on criterion $c$:

$$f_c(S) = \sum_{m \in \text{MODULES}} w_m \cdot \text{score}(da_m, c)$$

Output: set of non-dominated configurations (Pareto front).
Pareto-based MA does not depend on scenario to determine dominance —
module weights enter only in the computation of $f_c(S)$.

In [ ]:
def compute_criteria_scores(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    scores = {}
    for cfg in admissible:
        scores[cfg['id']] = {
            c: sum(w_mod[mod] * vetor_a.loc[cfg[mod], c] for mod in MODULES)
            for c in CRITERIA
        }
    return scores


def pareto_based_ma(admissible, scenario, vetor_a):
    crit_scores = compute_criteria_scores(admissible, scenario, vetor_a)
    ids = [cfg['id'] for cfg in admissible]
    dominated = set()

    for i, s1 in enumerate(ids):
        for s2 in ids:
            if s1 == s2:
                continue
            # Check if s1 is dominated by s2
            geq_all = all(crit_scores[s2][c] >= crit_scores[s1][c] for c in CRITERIA)
            gt_one  = any(crit_scores[s2][c] >  crit_scores[s1][c] for c in CRITERIA)
            if geq_all and gt_one:
                dominated.add(s1)
                break

    pareto_front = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_front, crit_scores


# Execute for all scenarios
print("Pareto-Based MA -- Pareto Front per scenario")
print("=" * 60)

pareto_results = {}
for sc_id, sc in SCENARIOS.items():
    front, crit_scores = pareto_based_ma(admissible, sc, vetor_a)
    pareto_results[sc_id] = {'front': front, 'crit_scores': crit_scores}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Pareto Front: {len(front)} of {len(admissible)} configurations")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'f_C1':>6} {'f_C2':>6} {'f_C3':>6} {'f_C4':>6} {'f_C5':>6}")
    print("-" * 70)
    for cfg in sorted(front, key=lambda x: x['id']):
        cs = crit_scores[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{cs['C1']:>6.3f} {cs['C2']:>6.3f} {cs['C3']:>6.3f} "
              f"{cs['C4']:>6.3f} {cs['C5']:>6.3f}")

#### Hierarchical Morphological Multicriteria Design (HMMD)

The system quality vector is defined as:

$$N(S) = (w(S);\, e(S))$$

Where:
- $w(S) = \min_{j_1 \neq j_2} \text{CCM}(da_{j_1}, da_{j_2})$ — minimum of pairwise compatibilities
- $e(S) = (\eta_1, \eta_2, \eta_3, \eta_4)$ — multiset vector counting DAs by priority level, with $\sum_r \eta_r = m$ (number of modules)

**Ordinal priorities of DAs per module:**
Calculated by score weighted by scenario criteria (D07):

$$\text{score}(da, sc) = \sum_{c \in \text{CRITERIA}} w_c(sc) \cdot \text{score}(da, c)$$

The ordinal position within the module is determined by ranking of this score
(position 1 = highest score). Ties receive the same position (D05).
Dimension $k = 4$ global for all modules (D06).

**Pareto-efficiency criterion over $N(S)$:**
$S$ is non-dominated if there does not exist $S'$ such that $N(S') \succ N(S)$,
where $\succ$ denotes lexicographic dominance over $(w(S); e(S))$.

**Optimization problem (Levin 2015, Section 2.2.7):**

$$\max\, e(S), \quad \max\, w(S), \quad \text{s.t.} \quad w(S) \geq 0.5$$

In [ ]:
def compute_priorities(scenario, vetor_a):
    w_crit = scenario['criteria_weights']
    priorities = {}

    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
            for da in das
        }
        # Ranking with ties at same position (dense ranking)
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        priorities[mod] = {da: rank_map[scores[da]] for da in das}

    return priorities


def compute_e(config, priorities, k=4):
    e = [0] * k
    for mod in MODULES:
        da = config[mod]
        rank = priorities[mod][da]
        if rank <= k:
            e[rank - 1] += 1
    return tuple(e)


def hmmd_dominates(n1, n2):
    w1, e1 = n1
    w2, e2 = n2

    # Comparison of w
    w_geq = w1 >= w2
    w_gt  = w1 > w2

    # Comparison of e by poset (lexicographic)
    e_geq = e1 >= e2  # tuple comparison -- lexicographic
    e_gt  = e1 > e2

    return (w_geq and e_geq) and (w_gt or e_gt)


def hmmd(admissible, scenario, vetor_a, k=4):
    priorities = compute_priorities(scenario, vetor_a)

    # Calculate N(S) for each configuration
    ns = {}
    for cfg in admissible:
        w = cfg['w']
        e = compute_e(cfg, priorities, k)
        ns[cfg['id']] = (w, e)

    # Identify non-dominated solutions
    dominated = set()
    ids = [cfg['id'] for cfg in admissible]
    for s1 in ids:
        for s2 in ids:
            if s1 == s2:
                continue
            if hmmd_dominates(ns[s2], ns[s1]):
                dominated.add(s1)
                break

    pareto_hmmd = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_hmmd, ns, priorities


# Execute for all scenarios
print("HMMD -- Pareto Front per scenario")
print("=" * 60)

hmmd_results = {}
for sc_id, sc in SCENARIOS.items():
    front, ns, priorities = hmmd(admissible, sc, vetor_a)
    hmmd_results[sc_id] = {'front': front, 'ns': ns, 'priorities': priorities}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"HMMD Front: {len(front)} of {len(admissible)} configurations")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'w(S)':>6} {'e(S)':<20} {'N(S)'}")
    print("-" * 75)
    for cfg in sorted(front, key=lambda x: x['id']):
        n = ns[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{n[0]:>6} {str(n[1]):<20} ({n[0]}, {n[1]})")

## Consolidated Experiments

Execution of the 4 selection methods on the admissible space (144 configurations).
Inputs: Vector A scale 1-5 (consolidated from 3 multi-LLM passes) · CCM scale {0, 0.5, 1.0}.

**Methods:**
- Linear weighted scoring: ranking by Q(S) descending
- Ideal point: ranking by distance ρ ascending to ideal point
- Pareto-based MA: non-dominated set over criterion vectors
- HMMD: non-dominated set over N(S) = (w(S); e(S))

In [ ]:
# Configuration map for quick lookup
cfg_map = {c['id']: c for c in admissible}

# Comparative table -- Scoring and Ideal Point per scenario
print("Section 4 -- Ranking Comparison")
print("=" * 60)

for sc_id, sc in SCENARIOS.items():
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Convert to rank dict by ID
    s_rank = {r['id']: r['rank'] for r in s_res}
    i_rank = {r['id']: r['rank'] for r in i_res}

    # Top 10 union (IDs appearing in any top 10)
    top_ids = list(dict.fromkeys(
        [r['id'] for r in s_res[:10]] +
        [r['id'] for r in i_res[:10]]
    ))

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<4} "
          f"{'Scoring':>8} {'Q':>7} {'Ideal':>7} {'rho':>7} {'Pareto':>8} {'HMMD':>6}")
    print("-" * 72)

    # Pareto front IDs for this scenario
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}
    hmmd_ids = {c['id'] for c in hmmd_results[sc_id]['front']}

    s_dict = {r['id']: r for r in s_res}
    i_dict = {r['id']: r for r in i_res}

    for sid in top_ids:
        cfg = cfg_map[sid]
        s_r = s_rank.get(sid, '-')
        i_r = i_rank.get(sid, '-')
        q = s_dict[sid]['Q']
        rho = i_dict[sid]['rho']
        pareto = 'Pareto' if sid in pareto_ids else ''
        hmmd_flag = 'HMMD' if sid in hmmd_ids else ''
        print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<4} "
              f"{s_r:>8} {q:>7.4f} {i_r:>7} {rho:>7.4f} {pareto:>8} {hmmd_flag:>6}")

## Comparative Analysis

Spearman correlation between methods, divergence identification,
and cross-cutting findings.

In [ ]:
print("Section 5 -- Comparative Analysis")
print("=" * 60)

# 5.1 Spearman Correlation -- Scoring vs Ideal Point
print("\n5.1 Spearman Correlation -- Scoring vs Ideal Point")
print("-" * 50)
print(f"{'Scenario':<8} {'rho':>8} {'p':>10}")
print("-" * 30)

for sc_id in SCENARIOS:
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Align rankings by ID
    ids = [r['id'] for r in s_res]
    s_ranks = [r['rank'] for r in s_res]
    i_rank_map = {r['id']: r['rank'] for r in i_res}
    i_ranks = [i_rank_map[sid] for sid in ids]

    rho, p = spearmanr(s_ranks, i_ranks)
    print(f"{sc_id:<8} {rho:>8.4f} {p:>10.4f}")

# 5.2 Spearman Correlation -- between scenarios (scoring)
print("\n5.2 Spearman Correlation -- Scoring between scenarios")
print("-" * 50)
sc_ids = list(SCENARIOS.keys())
print(f"{'':>8}", end='')
for sc in sc_ids:
    print(f"{sc:>8}", end='')
print()

for sc1 in sc_ids:
    print(f"{sc1:<8}", end='')
    r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
    for sc2 in sc_ids:
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        print(f"{rho:>8.4f}", end='')
    print()

# 5.3 Identified divergences
print("\n5.3 Identified divergences")
print("-" * 50)

for sc_id, sc in SCENARIOS.items():
    s_top10 = {r['id'] for r in scoring_results[sc_id][:10]}
    i_top10 = {r['id'] for r in ideal_results[sc_id][:10]}
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}

    # In Pareto but not in top 10 scoring
    pareto_not_scoring = pareto_ids - s_top10
    # In top 10 scoring but not in Pareto
    scoring_not_pareto = s_top10 - pareto_ids

    print(f"\n{sc_id} -- {sc['name']}")
    if pareto_not_scoring:
        print(f"  Pareto-efficient but outside top 10 scoring: "
              f"{', '.join(sorted(pareto_not_scoring))}")
        for sid in sorted(pareto_not_scoring):
            cfg = cfg_map[sid]
            cs = pareto_results[sc_id]['crit_scores'][sid]
            print(f"    {sid}: D={cfg['D']} I={cfg['I']} "
                  f"ID={cfg['ID']} R={cfg['R']}")
            print(f"    C1-C5 vector: "
                  f"{[round(float(cs[c]),3) for c in CRITERIA]}")
    if scoring_not_pareto:
        print(f"  Top 10 scoring but not Pareto-efficient: "
              f"{', '.join(sorted(scoring_not_pareto))}")
    if not pareto_not_scoring and not scoring_not_pareto:
        print("  No relevant divergences.")

# 5.4 Cross-cutting findings
print("\n5.4 Cross-cutting findings")
print("-" * 50)

# Ranking stability
rho_min = 1.0
rho_min_pair = ('', '')
for i, sc1 in enumerate(sc_ids):
    for sc2 in sc_ids[i+1:]:
        r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        if rho < rho_min:
            rho_min = rho
            rho_min_pair = (sc1, sc2)

print(f"Minimum Spearman between scenarios (scoring): "
      f"{rho_min:.4f} ({rho_min_pair[0]} x {rho_min_pair[1]})")

# Calculate mean and max rank per configuration
rank_stats = {}
for sid in [c['id'] for c in admissible]:
    ranks = [r['rank'] for sc in sc_ids
             for r in scoring_results[sc] if r['id'] == sid]
    rank_stats[sid] = {
        'mean': sum(ranks) / len(ranks),
        'max': max(ranks),
        'ranks': ranks
    }

# Sort by mean rank, tiebreak by max rank
sorted_by_robustness = sorted(
    rank_stats.items(),
    key=lambda x: (x[1]['mean'], x[1]['max'])
)

# Pareto and HMMD in all scenarios
pareto_todos = set.intersection(*[
    {c['id'] for c in pareto_results[sc]['front']}
    for sc in sc_ids
])
hmmd_todos = set.intersection(*[
    {c['id'] for c in hmmd_results[sc]['front']}
    for sc in sc_ids
])

print("Top 5 configurations by robustness (mean rank in scoring):")
print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
      f"{'Ranks SC1-SC4':<20} {'Mean':>6} {'Max':>5} {'Pareto':>8} {'HMMD':>6}")
print("-" * 80)
for sid, stats in sorted_by_robustness[:5]:
    cfg = cfg_map[sid]
    pareto_flag = 'Pareto' if sid in pareto_todos else ''
    hmmd_flag = 'HMMD' if sid in hmmd_todos else ''
    print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6} "
          f"{str(stats['ranks']):<20} {stats['mean']:>6.2f} {stats['max']:>5} "
          f"{pareto_flag:>8} {hmmd_flag:>6}")

# Identified trade-off
print(f"\nC5 trade-off vs C1-C4:")
if 'S50' in cfg_map:
    print(f"  S50 (D2, I2, ID2, R2) -- Pareto-efficient in all scenarios")
    cs_s50 = pareto_results['SC1']['crit_scores']['S50']
    print(f"  C1-C5 vector: {[round(float(cs_s50[c]), 3) for c in CRITERIA]}")
    print(f"  C5 higher than S46/S48, but C1-C4 lower.")
    print(f"  Does not appear in top 10 scoring in any scenario.")

## Diagnostic: Module Priorities

Priority analysis per scenario and module.

In [ ]:
# Diagnostic of priorities per scenario and module
for sc_id, sc in SCENARIOS.items():
    w_crit = sc['criteria_weights']
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'DA':<6} {'Module':<8} {'Score':>7} {'Rank':>6}")
    print("-" * 35)
    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: round(sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA), 4)
            for da in das
        }
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        for da in das:
            print(f"{da:<6} {mod:<8} {scores[da]:>7.4f} {rank_map[scores[da]]:>6}")

---
## References

Bader, K., Lussier, B., & Schön, W. (2017a). A fault tolerant architecture
for data fusion: A real application of Kalman filters for mobile robot
localization. *Robotics and Autonomous Systems*, *88*, 11-23.
https://doi.org/10.1016/j.robot.2016.12.001

Bader, K., Lussier, B., & Schön, W. (2017b). Fault tolerance from formal
analysis of a data fusion mechanism. In *2017 First IEEE International
Conference on Robotic Computing (IRC)* (pp. 69-74). IEEE.
https://doi.org/10.1109/IRC.2017.58

Ceccarelli, A., & Secci, F. (2023). RGB cameras failures and their effects
in autonomous driving applications. *IEEE Transactions on Dependable and
Secure Computing*, *20*(4), 2731-2745.
https://doi.org/10.1109/TDSC.2022.3156941

Conejo, C., Puig, V., Morcego, B., Navas, F., & Milanes, V. (2025).
Enhancing safety in autonomous vehicles using zonotopic LPV-EKF for fault
detection and isolation in state estimation. *Control Engineering Practice*,
*156*, 106192. https://doi.org/10.1016/j.conengprac.2024.106192

Elhoseny, M., Rao, D. D., Veerasamy, B. D., Alduaiji, N., Shreyas, J., &
Shukla, P. K. (2024). Deep learning algorithm for optimized sensor data
fusion in fault diagnosis and tolerance. *International Journal of
Computational Intelligence Systems*, *17*, 299.
https://doi.org/10.1007/s44196-024-00692-5

Emzivat, Y., Delahaye, B., Roux, O. H., & El Najjar, M. E. (2018). A formal
approach for the design of a dependable perception system for autonomous
vehicles. In *2018 IEEE Intelligent Vehicles Symposium (IV)* (pp. 1297-1304).
IEEE. https://doi.org/10.1109/IVS.2018.8500412

Erhan, L., Ndubuaku, M., Di Mauro, M., Song, W., Chen, M., Fortino, G.,
Bagdasar, O., & Liotta, A. (2021). Smart anomaly detection in sensor systems:
A multi-perspective review. *Information Fusion*, *67*, 64-79.
https://doi.org/10.1016/j.inffus.2020.10.001

Gao, Z., Cecati, C., & Ding, S. X. (2015a). A survey of fault diagnosis and
fault-tolerant techniques -- Part I: Fault diagnosis with model-based and
signal-based approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3757-3767. https://doi.org/10.1109/TIE.2015.2417570

Gao, Z., Cecati, C., & Ding, S. X. (2015b). A survey of fault diagnosis and
fault-tolerant techniques -- Part II: Fault diagnosis with knowledge-based
and hybrid/active approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3768-3774. https://doi.org/10.1109/TIE.2015.2419013

Goelles, T., Schlager, B., & Muckenhuber, S. (2020). Fault detection,
isolation, identification and recovery (FDIIR) methods for automotive
perception sensors including a detailed literature survey for lidar.
*Sensors*, *20*(13), 3662. https://doi.org/10.3390/s20133662

Grubmuller, S., Stettinger, G., Sotelo, M. A., & Watzenig, D. (2019).
Fault-tolerant environmental perception architecture for robust automated
driving. In *2019 IEEE International Conference on Connected Vehicles and
Expo (ICCVE)* (pp. 1-6). IEEE.
https://doi.org/10.1109/ICCVE45908.2019.8965112

Hao, Y., Ding, Y., Wang, G., Zhou, Y., & Jia, X. (2019). A fault-tolerant
data fusion method based on decentralized Kalman filter for redundant sensor
configuration. In *Proceedings of the 38th Chinese Control Conference*
(pp. 4486-4491). IEEE. https://doi.org/10.23919/ChiCC.2019.8856985

Hou, W., Li, W., & Li, P. (2023). Fault diagnosis of the autonomous driving
perception system based on information fusion. *Sensors*, *23*(11), 5110.
https://doi.org/10.3390/s23115110

Koopman, P., & Wagner, M. (2017). Autonomous vehicle safety: An
interdisciplinary challenge. *IEEE Intelligent Transportation Systems
Magazine*, *9*(1), 90-96. https://doi.org/10.1109/MITS.2016.2583491

Lee, J., Kang, H., Jeon, Y., Cho, J., Kim, S., & Jo, K. (2024). Sensor
fusion-based classification fault-tolerant system for detected objects in
autonomous cars. *IEEE Transactions on Intelligent Vehicles*.
https://doi.org/10.1109/TIV.2024.3361001

Mendonca, R., Rodrigues, J., Rodrigues, R., & Mendonca, R. (2023). Enhancing
the reliability of perception systems using N-version programming and
rejuvenation. In *2023 IEEE 28th Pacific Rim International Symposium on
Dependable Computing (PRDC)* (pp. 11-20). IEEE.
https://doi.org/10.1109/PRDC59308.2023.00012

Min, H., Fang, Y., Wu, X., Lei, X., Chen, S., Teixeira, R., Zhu, B., Zhao,
X., & Xu, Z. (2023). A fault diagnosis framework for autonomous vehicles with
sensor self-diagnosis. *Expert Systems with Applications*, *224*, 120002.
https://doi.org/10.1016/j.eswa.2023.120002

Nitsch, J., Itkina, M., Senanayake, R., Nieto, J., Schmidt, M., Siegwart,
R., Kochenderfer, M. J., & Cadena, C. (2021). Out-of-distribution detection
for automotive perception. In *2021 IEEE International Intelligent
Transportation Systems Conference (ITSC)* (pp. 2938-2943). IEEE.
https://doi.org/10.1109/ITSC48978.2021.9564545

Sinha, R., Schmerling, E., & Pavone, M. (2023). Closing the loop on runtime
monitors with fallback-safe MPC. *arXiv preprint arXiv:2309.08603*.
https://doi.org/10.48550/arXiv.2309.08603

Tian, H., Ding, W., Han, X., Wu, G., Guo, A., Zhang, J., Chen, W., Wei, J.,
& Zhang, T. (2025). Testing the fault-tolerance of multi-sensor fusion
perception in autonomous driving systems. *Proceedings of the ACM on
Software Engineering*, *2*(ISSTA), Article ISSTA035.
https://doi.org/10.1145/3728910

van Wyk, F., Wang, Y., Khojandi, A., & Masoud, N. (2020). Real-time sensor
anomaly detection and identification in automated vehicles. *IEEE
Transactions on Intelligent Transportation Systems*, *21*(3), 1264-1276.
https://doi.org/10.1109/TITS.2019.2906038

Werling, M., Faller, R., Betz, W., & Straub, D. (2025). Safety integrity
framework for automated driving. *arXiv preprint arXiv:2503.20544*.
https://doi.org/10.48550/arXiv.2503.20544

Levin, M. Sh. (2012). Morphological methods for design of modular systems:
A survey. *arXiv preprint arXiv:1201.1712*.

Levin, M. Sh. (2015). *Modular System Design and Evaluation*. Springer.

Ritchey, T. (2015). Principles of cross-consistency assessment in general
morphological modelling. *Acta Morphologica Generalis*, *4*(2).

Roy, B. (1996). *Multicriteria Methodology for Decision Aiding*.
Kluwer Academic Publishers.

### Evaluation Process — LLMs

#### Structured Debate — CCM E5 (scale {0, 0.5, 1.0})

| LLM | Link |
|---|---|
| Claude | https://claude.ai/share/5f265562-909f-4b60-af15-a6c738b30a71 |
| Gemini | https://gemini.google.com/share/ae86b4a76e6e |
| ChatGPT | https://chatgpt.com/share/69df8e68-2028-83e9-ba2b-c31e4ec4ac62 |
| DeepSeek | https://chat.deepseek.com/share/tltr4h05pjoxipg0xn |